<h1 style="color:DodgerBlue">Индивидальный проект</h1>

Система управления задачами проекта Project Task Manager

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


### Вариант задания № 23


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

В проекте реализована иерархия классов для управления задачами внутри проекта на языке C#.
Базовый класс `Task` описывает общую задачу (идентификатор, название, приоритет) и содержит
виртуальные методы выполнения, получения информации и переназначения задачи.
Три производных класса (`DelegateTask`, `TeamTask`, `ResearchTask`) расширяют базовый класс
собственными атрибутами и переопределяют методы.

Демонстрируются принципы ООП:
- **инкапсуляция** — закрытое поле `_priority`, свойства с проверкой значений;
- **наследование** — три класса наследуются от `Task`;
- **полиморфизм** — переопределение (override) и перегрузка (overload) методов;
- **обобщения (generic)** — класс `TaskRepository<T>` и обобщённый метод `GetHighestPriority<T>`.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// ===================== БАЗОВЫЙ КЛАСС =====================
public class Task
{
    private int _priority;                       // инкапсуляция

    // --- Основные атрибуты (по заданию) ---
    public int TaskId { get; }
    public string TaskName { get; set; }
    public int Priority
    {
        get => _priority;
        set => _priority = Math.Clamp(value, 1, 5);   // приоритет от 1 до 5
    }

    // --- Дополнительные атрибуты ---
    public string Description { get; set; }
    public DateTime CreatedDate { get; }
    public bool IsCompleted { get; protected set; }
    public string AssignedTo { get; protected set; }

    public Task(int taskId, string taskName, int priority,
                string description = "", string assignedTo = "Не назначен")
    {
        TaskId = taskId;
        TaskName = taskName;
        Priority = priority;
        Description = description;
        AssignedTo = assignedTo;
        CreatedDate = DateTime.Now;
        IsCompleted = false;
    }

    // --- Основные методы (virtual => можно переопределять) ---
    public virtual void MarkAsComplete()
    {
        IsCompleted = true;
        Console.WriteLine($"Задача '{TaskName}' (ID: {TaskId}) отмечена как выполненная.");
    }

    public virtual string GetTaskDetails()
    {
        string status = IsCompleted ? "Выполнена" : "В работе";
        return $"[{TaskId}] {TaskName} | Приоритет: {Priority} | Статус: {status} | " +
               $"Исполнитель: {AssignedTo} | Создана: {CreatedDate:dd.MM.yyyy}";
    }

    public virtual void ReassignTo(string newMember)
    {
        AssignedTo = newMember;
        Console.WriteLine($"Задача '{TaskName}' переназначена на: {newMember}");
    }

    // --- ПЕРЕГРУЗКА методов (overload) ---
    public virtual void MarkAsComplete(string comment)
    {
        MarkAsComplete();
        Console.WriteLine($"  Комментарий: {comment}");
    }

    public void ReassignTo(string newMember, string reason)
    {
        ReassignTo(newMember);
        Console.WriteLine($"  Причина: {reason}");
    }

    // --- Дополнительные методы ---
    public bool IsHighPriority() => Priority >= 4;

    public void ChangePriority(int newPriority)
    {
        Priority = newPriority;
        Console.WriteLine($"Приоритет задачи '{TaskName}' изменён на {Priority}");
    }
}

// ===================== ПРОИЗВОДНЫЙ КЛАСС 1 =====================
public class DelegateTask : Task
{
    public DateTime DueDate { get; set; }
    public string DelegatedBy { get; set; }

    public DelegateTask(int id, string name, int priority, DateTime dueDate,
                        string delegatedBy = "Руководитель")
        : base(id, name, priority)
    {
        DueDate = dueDate;
        DelegatedBy = delegatedBy;
    }

    // Переопределение: добавляем дату выполнения в сообщение
    public override void MarkAsComplete()
    {
        IsCompleted = true;
        string timing = DateTime.Now.Date <= DueDate.Date ? "в срок" : "с опозданием";
        Console.WriteLine($"[Делегированная] Задача '{TaskName}' выполнена {timing}. " +
                          $"Срок выполнения: {DueDate:dd.MM.yyyy}");
    }

    public override string GetTaskDetails() =>
        base.GetTaskDetails() + $" | Делегировал: {DelegatedBy} | Срок: {DueDate:dd.MM.yyyy}";

    public int DaysLeft() => (DueDate.Date - DateTime.Now.Date).Days;

    public bool IsOverdue() => !IsCompleted && DateTime.Now.Date > DueDate.Date;
}

// ===================== ПРОИЗВОДНЫЙ КЛАСС 2 =====================
public class TeamTask : Task
{
    public string TeamName { get; set; }
    public List<string> Members { get; } = new List<string>();

    public TeamTask(int id, string name, int priority, string teamName)
        : base(id, name, priority)
    {
        TeamName = teamName;
    }

    public void AddMember(string member)
    {
        if (!Members.Contains(member))
            Members.Add(member);
    }

    public int GetTeamSize() => Members.Count;

    // Переопределение: указываем нового члена команды
    public override void ReassignTo(string newMember)
    {
        if (!Members.Contains(newMember))
        {
            Console.WriteLine($"[Командная] '{newMember}' не входит в команду '{TeamName}'. Переназначение отклонено.");
            return;
        }
        AssignedTo = newMember;
        Console.WriteLine($"[Командная] Задача '{TaskName}' команды '{TeamName}' " +
                          $"переназначена на нового члена команды: {newMember}");
    }

    public override string GetTaskDetails() =>
        base.GetTaskDetails() + $" | Команда: {TeamName} ({GetTeamSize()} чел.)";
}

// ===================== ПРОИЗВОДНЫЙ КЛАСС 3 =====================
public class ResearchTask : Task
{
    public string DataSource { get; set; }
    public string ResearchField { get; set; }
    public int ProgressPercent { get; private set; }

    public ResearchTask(int id, string name, int priority, string dataSource,
                        string researchField = "Общая")
        : base(id, name, priority)
    {
        DataSource = dataSource;
        ResearchField = researchField;
    }

    public void UpdateProgress(int percent)
    {
        ProgressPercent = Math.Clamp(percent, 0, 100);
        Console.WriteLine($"Прогресс исследования '{TaskName}': {ProgressPercent}%");
    }

    // Переопределение: показываем источники данных
    public override string GetTaskDetails() =>
        base.GetTaskDetails() +
        $" | Область: {ResearchField} | Исходные данные: {DataSource} | Прогресс: {ProgressPercent}%";
}

// ===================== GENERIC КЛАССЫ =====================
public class TaskRepository<T> where T : Task
{
    private readonly List<T> _items = new List<T>();

    public int Count => _items.Count;

    public void Add(T task)
    {
        _items.Add(task);
        Console.WriteLine($"В хранилище добавлена задача: {task.TaskName}");
    }

    public bool Remove(T task) => _items.Remove(task);

    public T GetById(int id) => _items.FirstOrDefault(t => t.TaskId == id);

    public IEnumerable<T> Find(Func<T, bool> predicate) => _items.Where(predicate);

    public void PrintAll()
    {
        foreach (var t in _items)
            Console.WriteLine(t.GetTaskDetails());   // полиморфный вызов
    }
}

public static class TaskUtils
{
    // Generic-метод
    public static T GetHighestPriority<T>(IEnumerable<T> tasks) where T : Task
        => tasks.OrderByDescending(t => t.Priority).FirstOrDefault();
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
Console.WriteLine("=== 1. Создание объектов ===");
var delegateTask = new DelegateTask(1, "Подготовить отчёт", 4, DateTime.Now.AddDays(3), "Менеджер Алим");
var teamTask = new TeamTask(2, "Разработка модуля оплаты", 5, "Backend-команда");
var researchTask = new ResearchTask(3, "Анализ пользовательских данных", 3, "База данных клиентов 2025", "Аналитика");

teamTask.AddMember("Хамид");
teamTask.AddMember("Сара");
researchTask.UpdateProgress(40);

Console.WriteLine("\n=== 2. Полиморфизм: один список базового типа ===");
List<Task> tasks = new List<Task> { delegateTask, teamTask, researchTask };
foreach (Task t in tasks)
    Console.WriteLine(t.GetTaskDetails());

Console.WriteLine("\n=== 3. Переопределение MarkAsComplete (DelegateTask) ===");
delegateTask.MarkAsComplete();

Console.WriteLine("\n=== 4. Переопределение ReassignTo (TeamTask) ===");
teamTask.ReassignTo("Хамид");
teamTask.ReassignTo("Неизвестный");

Console.WriteLine("\n=== 5. Перегрузка методов ===");
researchTask.ReassignTo("Ахмад");
researchTask.ReassignTo("Ахмад", "Необходим опыт в аналитике");
researchTask.MarkAsComplete("Все данные обработаны");

Console.WriteLine("\n=== 6. Generic-класс TaskRepository<T> ===");
var repo = new TaskRepository<Task>();
foreach (var t in tasks) repo.Add(t);
repo.PrintAll();

var teamRepo = new TaskRepository<TeamTask>();
teamRepo.Add(teamTask);

Console.WriteLine("\n=== 7. Generic-метод ===");
var top = TaskUtils.GetHighestPriority(tasks);
Console.WriteLine($"Самая приоритетная задача: {top.TaskName} (приоритет {top.Priority})");

Console.WriteLine("\n=== 8. Дополнительные методы ===");
Console.WriteLine($"Осталось дней: {delegateTask.DaysLeft()}");
Console.WriteLine($"Высокий приоритет у командной задачи? {teamTask.IsHighPriority()}");

=== 1. Создание объектов ===
Прогресс исследования 'Анализ пользовательских данных': 40%

=== 2. Полиморфизм: один список базового типа ===
[1] Подготовить отчёт | Приоритет: 4 | Статус: В работе | Исполнитель: Не назначен | Создана: 02.11.2026 | Делегировал: Менеджер Алим | Срок: 05.11.2026
[2] Разработка модуля оплаты | Приоритет: 5 | Статус: В работе | Исполнитель: Не назначен | Создана: 02.11.2026 | Команда: Backend-команда (2 чел.)
[3] Анализ пользовательских данных | Приоритет: 3 | Статус: В работе | Исполнитель: Не назначен | Создана: 02.11.2026 | Область: Аналитика | Исходные данные: База данных клиентов 2025 | Прогресс: 40%

=== 3. Переопределение MarkAsComplete (DelegateTask) ===
[Делегированная] Задача 'Подготовить отчёт' выполнена в срок. Срок выполнения: 05.11.2026

=== 4. Переопределение ReassignTo (TeamTask) ===
[Командная] Задача 'Разработка модуля оплаты' команды 'Backend-команда' переназначена на нового члена команды: Хамид
[Командная] 'Неизвестный' не входит в команд